In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys
import pickle

# Configuration
headers = {'User-Agent': 'DataEngineeringStudentBot/1.0'}
base_url = "https://www.staticstools.eu"
root_url = f"{base_url}/en/"
all_dataframes = {}

print("✅ Environment ready.")

In [ ]:
print("🔍 STEP 1: Bootstrapping profile list...")

try:
    res = requests.get(root_url, headers=headers, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')

    links = soup.select('a[href*="profile-"]')
    profile_list = []

    for link in links:
        p_path = link['href']
        # Robust Name Extraction: Get name from link text or URL slug
        p_name = link.text.strip()
        if not p_name:
            # Extracts 'shs' from '/en/profile-shs'
            p_name = p_path.split('profile-')[-1].split('/')[0].upper()

        p_url = f"{base_url}/{p_path.lstrip('/')}"

        try:
            p_res = requests.get(p_url, headers=headers, timeout=10)
            p_soup = BeautifulSoup(p_res.text, 'html.parser')
            select_tag = p_soup.find('select', {'name': 'profile'})

            if select_tag:
                s_ids = [opt['value'] for opt in select_tag.find_all(
                    'option') if opt.get('value')]
                profile_list.append({
                    'name': p_name,
                    'url': p_url,
                    'ids': s_ids,
                    'count': len(s_ids)
                })
                print(f"  📂 Found {p_name:10} | {len(s_ids):>3} sections.")
        except Exception as e:
            print(f"  ⚠️ Skipping {p_name}: {e}")

    df_profiles = pd.DataFrame(profile_list)
    total_expected = df_profiles['count'].sum()
    print(
        f"\n✅ Bootstrap complete. Total sections to scrape: {total_expected}")

except Exception as e:
    print(f"❌ Critical Error during bootstrap: {e}")

In [ ]:
def get_section_data_with_meta2(url):
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code != 200:
            return None, None

        soup = BeautifulSoup(res.text, 'html.parser')
        section_values = {}
        section_meta = {}

        cells = soup.find_all('td')
        for cell in cells:
            text = cell.text.strip()
            description = cell.get('title', '').strip()

            if '=' in text:
                parts = text.split('=')
                if len(parts) == 2:
                    prop_name = parts[0].strip()
                    val_raw_parts = parts[1].strip().split(' ')

                    val_str = val_raw_parts[0]
                    unit_str = val_raw_parts[1] if len(
                        val_raw_parts) > 1 else ""

                    try:
                        clean_val = val_str.replace(',', '.')
                        section_values[prop_name] = float(clean_val)
                    except:
                        section_values[prop_name] = val_str

                    section_meta[prop_name] = (description, unit_str)

        return section_values, section_meta
    except:
        return None, None


print("✅ Extraction function defined.")

In [ ]:
def get_section_data_with_meta(url):
    try:
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code != 200:
            return None, None

        soup = BeautifulSoup(res.text, 'html.parser')
        section_values = {}
        section_meta = {}

        # --- THIS WAS MISSING ---
        cells = soup.find_all('td')
        # ------------------------

        for cell in cells:
            text = cell.text.strip()
            description = cell.get('title', '').strip()

            if '=' in text:
                # Split by '=' and clean whitespace
                parts = [p.strip() for p in text.split('=')]

                if len(parts) >= 2:
                    # Last part is "Value Unit" (e.g., "5.30E+7 mm4")
                    val_and_unit = parts[-1]
                    prop_names = parts[:-1]

                    val_raw_parts = val_and_unit.split(' ')
                    val_str = val_raw_parts[0]
                    unit_str = val_raw_parts[1] if len(
                        val_raw_parts) > 1 else ""

                    try:
                        clean_val = float(val_str.replace(',', '.'))
                    except:
                        clean_val = val_str

                    for prop_name in prop_names:
                        # --- SMART META FIX ---
                        # Prevent "y-y axis" description on "Iz" properties
                        final_desc = description
                        if 'z' in prop_name.lower() and 'y' in final_desc.lower():
                            final_desc = final_desc.replace(
                                'y-y', 'z-z').replace('y axis', 'z axis')

                        section_values[prop_name] = clean_val
                        section_meta[prop_name] = (final_desc, unit_str)

        return section_values, section_meta
    except Exception as e:
        # Temporary print to see if it's failing for other reasons
        # print(f"Error at {url}: {e}")
        return None, None

In [ ]:
# print(f"🚀 STEP 2: Starting Master Scrape...")
# current_count = 0

# for _, row in df_profiles.iterrows():
#     p_name = row['name']
#     s_ids = row['ids']
#     current_profile_rows = []
#     profile_type_meta = {}

#     for sid in s_ids:
#         current_count += 1
#         sys.stdout.write(
#             f"\r   [{current_count}/{total_expected}] Processing {p_name} -> {sid}...")
#         sys.stdout.flush()

#         # Fix: Ensure the slug is lowercase for the URL
#         family_slug = p_name.lower()
#         detail_url = f"{base_url}/en/profile-{family_slug}/{sid}/mm/show"

#         data, meta = get_section_data_with_meta(detail_url)

#         if data:
#             data['Section_ID'] = sid
#             current_profile_rows.append(data)
#             if meta:
#                 profile_type_meta.update(meta)

#         time.sleep(0.1)  # Essential to avoid being rate-limited

#     if current_profile_rows:
#         df_temp = pd.DataFrame(current_profile_rows)
#         cols = ['Section_ID'] + \
#             [c for c in df_temp.columns if c != 'Section_ID']
#         df_final = df_temp[cols].copy()

#         profile_type_meta['Section_ID'] = ('Unique Section Identifier', 'text')
#         df_final.attrs['column_meta'] = profile_type_meta

#         all_dataframes[f"df_{p_name}"] = df_final
#         print(f" ✅ {p_name} complete.")
#     else:
#         print(f" ❌ ERROR: No data found for {p_name}. Check URL: {detail_url}")

# # Save the final product
# with open('steel_profiles_full_data.pkl', 'wb') as f:
#     pickle.dump(all_dataframes, f)

# print(
#     f"\n🏆 SCRAPE COMPLETE! {len(all_dataframes)} DataFrames saved to pickle.")

# LOAD

In [56]:
import pickle

# Load the full dictionary with attributes preserved
with open('steel_profiles_full_data.pkl', 'rb') as f:
    all_dataframes = pickle.load(f)

print(f"✅ Data loaded successfully.")
print(f"Found {len(all_dataframes)} DataFrames in the collection.")

✅ Data loaded successfully.
Found 22 DataFrames in the collection.


In [57]:
# import pandas as pd

# # Load CSV without treating the first row as headers
# df_rhs_csv = pd.read_csv('RHS.csv', header=None)


In [ ]:
# 1. Capture existing metadata and prepare the DataFrame
df_updated = all_dataframes['df_RHS'].copy()
current_meta = df_updated.attrs.get('column_meta', {}).copy()

# 2. Rename 't' to 'tw' in the data and the metadata
df_updated.rename(columns={'t': 'tw'}, inplace=True)
if 't' in current_meta:
    current_meta['tw'] = current_meta.pop('t')

# 3. Define 'r_in' and 'r_out'
# Based on your logic: r_out = r_in * 1.5 (rounded to 1 decimal place)
# We assume 'r_in' already exists or is derived from the previous 'r'
if 'r' in df_updated.columns:
    df_updated.rename(columns={'r': 'r_in'}, inplace=True)
    if 'r' in current_meta:
        current_meta['r_in'] = current_meta.pop('r')

df_updated['r_out'] = (df_updated['r_in'] * 1.5).round(1)

# 4. Inject/Update metadata for the new radius definitions
current_meta['r_in'] = ('Radius of inner fillet', 'mm')
current_meta['r_out'] = ('Radius of outer fillet', 'mm')

# 5. Re-attach metadata and update the collection
df_updated.attrs['column_meta'] = current_meta
all_dataframes['df_RHS'] = df_updated

print("✅ Update complete: 't' renamed to 'tw', 'r_out' calculated, and metadata preserved.")

✅ Update complete: 't' renamed to 'tw', 'r_out' calculated, and metadata preserved.


In [59]:
# 1. Capture and prepare metadata
current_meta = all_dataframes['df_SHS'].attrs.get('column_meta', {}).copy()

# 2. Extract the SHS dataframe and rename 't' to 'tw' immediately
shs = all_dataframes['df_SHS'].copy()
shs.rename(columns={'t': 'tw'}, inplace=True)

# Update metadata key from 't' to 'tw'
if 't' in current_meta:
    current_meta['tw'] = current_meta.pop('t')

# 3. Calculate new radii based on the updated 'tw' column
# r_in = tw
# r_out = 1.5 * r_in (rounded to 1 decimal)
shs['r_in'] = shs['tw'].astype(float)
shs['r_out'] = (shs['r_in'] * 1.5).round(1)

# 4. Drop the old 'r' column and its metadata
shs.drop(columns=['r'], inplace=True, errors='ignore')
current_meta.pop('r', None)

# 5. Add new metadata descriptions
current_meta['r_out'] = ('Radius of outer fillet', 'mm')
current_meta['r_in'] = ('Radius of inner fillet', 'mm')

# 6. Re-attach and save back to the collection
shs.attrs['column_meta'] = current_meta
all_dataframes['df_SHS'] = shs

# Quick Verification
print(f"✅ Updated df_SHS: Renamed 't' to 'tw', added r_in and r_out.")
print(
    f"Sample: tw={shs.iloc[0]['tw']}, r_in={shs.iloc[0]['r_in']}, r_out={shs.iloc[0]['r_out']}")

✅ Updated df_SHS: Renamed 't' to 'tw', added r_in and r_out.
Sample: tw=2.6, r_in=2.6, r_out=3.9


In [60]:
# # 1. Capture existing metadata before it's lost in the merge
# current_meta = all_dataframes['df_RHS'].attrs.get('column_meta', {}).copy()

# # 2. Prepare the radius data from CSV (Cols 0=ID, 5=r_out, 6=r_in)
# df_radius = df_rhs_csv[[0, 5, 6]].rename(
#     columns={0: 'Section_ID', 5: 'r_out', 6: 'r_in'})

# # 3. Clean IDs and Merge
# # We strip the '+' from Section_ID to ensure they match the CSV format
# df_updated = all_dataframes['df_RHS'].copy()
# df_updated['Section_ID'] = df_updated['Section_ID'].str.replace(
#     '+', '', regex=False)
# df_updated = df_updated.merge(df_radius, on='Section_ID', how='left')

# # 4. Drop the old 'r' column and its metadata
# df_updated.drop(columns=['r'], inplace=True, errors='ignore')
# current_meta.pop('r', None)

# # 5. Inject new metadata for the added columns
# current_meta['r_out'] = ('Radius of outer fillet', 'mm')
# current_meta['r_in'] = ('Radius of inner fillet', 'mm')

# # 6. Re-attach the metadata to the new DataFrame and update the collection
# df_updated.attrs['column_meta'] = current_meta
# all_dataframes['df_RHS'] = df_updated

# print("✅ Update complete: Columns added, 'r' dropped, and .attrs preserved.")

In [61]:
import pandas as pd

# 1. Define the targets for suffixing
radius_targets = ['iy', 'iz', 'iu', 'iv', 'iw']

for key in list(all_dataframes.keys()):
    df = all_dataframes[key]

    # --- Part A: Handle the df_SHS 'a' -> 'h' fix ---
    if key == 'df_SHS' and 'a' in df.columns:
        # Update DataFrame
        df.rename(columns={'a': 'h'}, inplace=True)
        # Update Attributes
        if 'column_meta' in df.attrs:
            meta = df.attrs['column_meta']
            if 'a' in meta:
                meta['h'] = meta.pop('a')
        print(f"✅ Fixed {key}: Renamed 'a' to 'h'")

    # --- Part B: Suffix the Radii (iy -> iy_radius) ---
    cols_to_rename = {
        col: f"{col}_radius" for col in df.columns if col in radius_targets}

    if cols_to_rename:
        # Update DataFrame columns
        df.rename(columns=cols_to_rename, inplace=True)

        # Update Metadata Attributes keys
        if 'column_meta' in df.attrs:
            meta = df.attrs['column_meta']
            for old_col, new_col in cols_to_rename.items():
                if old_col in meta:
                    meta[new_col] = meta.pop(old_col)

        print(f"✅ Fixed {key}: Suffixed {list(cols_to_rename.keys())}")

print("\n🚀 All DataFrames and Attributes updated successfully.")

✅ Fixed df_IPN: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_IPE: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_IPEA: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_IPEAA: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_IPEO: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HE: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HEA: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HEAA: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HEB: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HEM: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HD: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_HL: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_UPN: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_UE: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_UPE: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_UAP: Suffixed ['iy', 'iz', 'iw']
✅ Fixed df_LE: Suffixed ['iy', 'iz', 'iu', 'iv']
✅ Fixed df_LU: Suffixed ['iy', 'iz', 'iu', 'iv']
✅ Fixed df_T: Suffixed ['iy', 'iz']
✅ Fixed df_SHS: Renamed 'a' to 'h'
✅ Fixed df_SHS: Suffixed ['iy', 'iz']
✅ Fixed df_RHS: Suffixed ['iy', 'iz']
✅ Fixed df_CHS: Suffixed ['iy', 'iz']

🚀 All DataF

In [62]:
# Update the specific metadata across all tables that have these columns
for key, df in all_dataframes.items():
    if 'column_meta' in df.attrs:
        meta = df.attrs['column_meta']

        # Update iw_radius description
        if 'iw_radius' in meta:
            meta['iw_radius'] = (
                'Radius of gyration of the warping constant', 'mm')

        # Update α (alpha) unit to [deg]
        if 'α' in meta:
            desc, _ = meta['α']
            meta['α'] = (desc, 'deg')

print(
    "✅ Metadata updated: 'iw_radius' description and 'α' units [deg] are now set.")

✅ Metadata updated: 'iw_radius' description and 'α' units [deg] are now set.


In [63]:
# Mapping Dictionary for logic
rename_logic = {
    # I & H Profiles
    'df_IPE':  {'r1': 'r_root'},
    'df_IPEA': {'r1': 'r_root'},
    'df_IPEAA': {'r1': 'r_root'},
    'df_IPEO': {'r1': 'r_root'},
    'df_HE':   {'r1': 'r_root'},
    'df_HEA':  {'r1': 'r_root'},
    'df_HEAA': {'r1': 'r_root'},
    'df_HEB':  {'r1': 'r_root'},
    'df_HEM':  {'r1': 'r_root'},
    'df_HD':   {'r1': 'r_root'},
    'df_HL':   {'r1': 'r_root'},

    # Specific I-Section (IPN) - Now includes description update
    'df_IPN':  {'r1': 'r_root', 'r2': 'r_toe'},

    # U-Sections
    'df_UPN':  {'r1': 'r_root', 'r2': 'r_toe'},
    'df_UE':   {'r1': 'r_root', 'r2': 'r_toe'},
    'df_UPE':  {'r':  'r_root'},
    'df_UAP':  {'r':  'r_root'},

    # L-Sections (Angles)
    'df_LE':   {'r1': 'r_root', 'r2': 'r_toe'},
    'df_LU':   {'r1': 'r_root', 'r2': 'r_toe'},

    # T-Sections (The Special Case)
    'df_T':    {'r': 'r_root', 'r1': 'r_toe', 'r2': 'r_web'}
}

# Standardized descriptions to apply during renaming
standard_descriptions = {
    'r_root': ('Radius of root fillet', 'mm'),
    'r_toe':  ('Radius of flange toe fillet', 'mm'),
    'r_web':  ('Radius of web toe fillet', 'mm')
}

for key, mapping in rename_logic.items():
    if key in all_dataframes:
        df = all_dataframes[key]

        # 1. Rename DataFrame Columns physically
        df.rename(columns=mapping, inplace=True)

        # 2. Update the Metadata dictionary in .attrs
        if 'column_meta' in df.attrs:
            new_meta = {}
            for col, val in df.attrs['column_meta'].items():
                # Get the new column name if it was mapped
                new_col_name = mapping.get(col, col)

                # Apply standard descriptions for root, toe, and web radii
                # This specifically fixes the df_IPN 'toe radius' -> 'Radius of flange toe fillet'
                if new_col_name in standard_descriptions:
                    new_meta[new_col_name] = standard_descriptions[new_col_name]
                else:
                    new_meta[new_col_name] = val

            df.attrs['column_meta'] = new_meta
            print(
                f"✅ Processed {key}: Columns renamed and metadata standardized.")

print("\n🚀 All radii standardized. You can now safely run your SQL migration.")

✅ Processed df_IPE: Columns renamed and metadata standardized.
✅ Processed df_IPEA: Columns renamed and metadata standardized.
✅ Processed df_IPEAA: Columns renamed and metadata standardized.
✅ Processed df_IPEO: Columns renamed and metadata standardized.
✅ Processed df_HE: Columns renamed and metadata standardized.
✅ Processed df_HEA: Columns renamed and metadata standardized.
✅ Processed df_HEAA: Columns renamed and metadata standardized.
✅ Processed df_HEB: Columns renamed and metadata standardized.
✅ Processed df_HEM: Columns renamed and metadata standardized.
✅ Processed df_HD: Columns renamed and metadata standardized.
✅ Processed df_HL: Columns renamed and metadata standardized.
✅ Processed df_IPN: Columns renamed and metadata standardized.
✅ Processed df_UPN: Columns renamed and metadata standardized.
✅ Processed df_UE: Columns renamed and metadata standardized.
✅ Processed df_UPE: Columns renamed and metadata standardized.
✅ Processed df_UAP: Columns renamed and metadata stand

#### T

In [64]:
import pandas as pd

# 1. Reference the dataframe
df_t = all_dataframes['df_T']
current_meta = df_t.attrs.get('column_meta', {}).copy()

# 2. Calculate New Geometric Columns
# Note: Ensure 'b' exists in the dataframe before this calculation
# df_t['h'] = df_t['Section_ID'].str.extract(r'(\d+)').astype(float)
# df_t['b'] = df_t['h']
df_t['tw'] = (df_t['b'] / 10.0) + 1.0
df_t['tf'] = df_t['tw']

# 3. Drop redundant columns 's' and 't'
# errors='ignore' ensures the code doesn't crash if they were already removed
df_t.drop(columns=['s', 't'], inplace=True, errors='ignore')

# 4. Prepare New Metadata and Clean Old Keys
new_meta_entries = {
    'h': ('Depth of section', 'mm'),
    'b': ('Width of section', 'mm'),
    'tw': ('Web thickness', 'mm'),
    'tf': ('Flange thickness', 'mm')
}

# Remove 's' and 't' from the metadata dictionary
current_meta.pop('s', None)
current_meta.pop('t', None)

# Update with the new entries
current_meta.update(new_meta_entries)

# 5. Re-attach the cleaned metadata
df_t.attrs['column_meta'] = current_meta

print("✅ df_T updated: 'tw' and 'tf' calculated. Redundant columns 's' and 't' removed.")

✅ df_T updated: 'tw' and 'tf' calculated. Redundant columns 's' and 't' removed.


In [65]:
# import pandas as pd

# # 1. Reference the dataframe
# df_t = all_dataframes['df_T']

# # 2. Calculate New Geometric Columns
# # Added 'r' before the string to fix the SyntaxWarning
# # df_t['h'] = df_t['Section_ID'].str.extract(r'(\d+)').astype(float)
# # df_t['b'] = df_t['h']
# df_t['tw'] = (df_t['b'] / 10.0) + 1.0
# df_t['tf'] = df_t['tw']

# # 3. Define the New Metadata
# new_meta_entries = {
#     'h': ('Depth of section', 'mm'),
#     'b': ('Width of section', 'mm'),
#     'tw': ('Web thickness', 'mm'),
#     'tf': ('Flange thickness', 'mm')
# }

# # 4. Update the .attrs dictionary
# if 'column_meta' in df_t.attrs:
#     df_t.attrs['column_meta'].update(new_meta_entries)

# print("✅ df_T updated: columns calculated and metadata synced (SyntaxWarning resolved).")

In [66]:
for key, df in all_dataframes.items():
    if 'Section_ID' in df.columns:
        # Replace "+" with an empty string
        df['Section_ID'] = df['Section_ID'].str.replace('+', '', regex=False)

        # Optional: Clean up any double spaces that might have been left behind
        df['Section_ID'] = df['Section_ID'].str.strip()

        print(f"✅ Cleaned Section_ID in {key}")

✅ Cleaned Section_ID in df_IPN
✅ Cleaned Section_ID in df_IPE
✅ Cleaned Section_ID in df_IPEA
✅ Cleaned Section_ID in df_IPEAA
✅ Cleaned Section_ID in df_IPEO
✅ Cleaned Section_ID in df_HE
✅ Cleaned Section_ID in df_HEA
✅ Cleaned Section_ID in df_HEAA
✅ Cleaned Section_ID in df_HEB
✅ Cleaned Section_ID in df_HEM
✅ Cleaned Section_ID in df_HD
✅ Cleaned Section_ID in df_HL
✅ Cleaned Section_ID in df_UPN
✅ Cleaned Section_ID in df_UE
✅ Cleaned Section_ID in df_UPE
✅ Cleaned Section_ID in df_UAP
✅ Cleaned Section_ID in df_LE
✅ Cleaned Section_ID in df_LU
✅ Cleaned Section_ID in df_T
✅ Cleaned Section_ID in df_SHS
✅ Cleaned Section_ID in df_RHS
✅ Cleaned Section_ID in df_CHS


In [67]:
def clean_metadata(meta_dict):
    cleaned = {}
    for key, (desc, unit) in meta_dict.items():
        # Fix the Z-axis/Y-axis confusion
        if key.lower().startswith('z') and 'y-axis' in desc.lower():
            desc = desc.replace('y-axis', 'z-axis')

        # Fix the spelling of length
        desc = desc.replace('lenght', 'length')

        # Standardize alpha
        if key == 'α':
            key = 'alpha'

        cleaned[key] = (desc, unit)
    return cleaned


# Apply it to your scrape
for df_name in all_dataframes:
    meta = all_dataframes[df_name].attrs.get('column_meta', {})
    all_dataframes[df_name].attrs['column_meta'] = clean_metadata(meta)

#### LU & LE

In [68]:
# List of the angle section dataframes to update
angle_keys = ['df_LE', 'df_LU']

for key in angle_keys:
    # 1. Reference the dataframe and copy metadata
    df = all_dataframes[key]
    current_meta = df.attrs.get('column_meta', {}).copy()

    # 2. Rename 't' to 'tw' in the DataFrame
    df.rename(columns={'t': 'tw'}, inplace=True)

    # 3. Update the metadata dictionary
    if 't' in current_meta:
        # Move description from 't' to 'tw'
        current_meta['tw'] = current_meta.pop('t')
    else:
        # Fallback if 't' wasn't present
        current_meta['tw'] = ('Web thickness', 'mm')

    # 4. Re-attach the cleaned metadata
    df.attrs['column_meta'] = current_meta

    print(f"✅ Updated {key}: Renamed 't' to 'tw' and synced metadata.")

✅ Updated df_LE: Renamed 't' to 'tw' and synced metadata.
✅ Updated df_LU: Renamed 't' to 'tw' and synced metadata.


#### CHS

In [69]:
# 1. Capture metadata and DataFrame
df_chs = all_dataframes['df_CHS']
current_meta = df_chs.attrs.get('column_meta', {}).copy()

# 2. Rename Column 'T' to 'tw' in the DataFrame
df_chs.rename(columns={'T': 'tw'}, inplace=True)

# 3. Handle Metadata updates
# Update 'D' description
current_meta['D'] = ('Outer diameter of section', 'mm')

# Move 'T' description to 'tw'
if 'T' in current_meta:
    current_meta['tw'] = current_meta.pop('T')
else:
    current_meta['tw'] = ('Web thickness', 'mm')

# 4. Re-attach updated metadata
df_chs.attrs['column_meta'] = current_meta

print("✅ Updated df_CHS: 'D' description updated and 'T' renamed to 'tw'.")

✅ Updated df_CHS: 'D' description updated and 'T' renamed to 'tw'.


#### Attributes

In [70]:
from pprint import pprint

for key, df in all_dataframes.items():
    print(f"--- Key: {key} ---")
    pprint(df.attrs['column_meta'])
    print("\n")

--- Key: df_IPN ---
{'A': ('Area of section', 'mm2'),
 'AL': ('Painting surface per unit length', 'm2.m-1'),
 'G': ('Mass per unit length', 'kg.m-1'),
 'It': ('Torsion constant', 'mm4'),
 'Iw': ('Warping constant', 'mm6'),
 'Iy': ('Second moment of area about the y-y axis', 'mm4'),
 'Iz': ('Second moment of area about the z-z axis', 'mm4'),
 'Section_ID': ('Unique Section Identifier', 'text'),
 'Sy': ('First moment of area about the y-y axis', 'mm3'),
 'Sz': ('First moment of area about the z-z axis', 'mm3'),
 'Wy,pl': ('Plastic modulus of section about the y-y axis', 'mm3'),
 'Wy1': ('Elastic section modulus', 'mm3'),
 'Wz,pl': ('Plastic Modulus of section about the z-z axis', 'mm3'),
 'Wz1': ('Elastic Modulus of section about the z-z axis to point 1', 'mm3'),
 'b': ('Width of section', 'mm'),
 'd': ('Depth of straight portion of web', 'mm'),
 'h': ('Depth of section', 'mm'),
 'ipc': ('Polar radius of gyration', 'mm'),
 'iw_radius': ('Radius of gyration of the warping constant', 'mm')

In [71]:
# Create a list to store unique pairs
unique_pairs = []
# Use a set to keep track of what we've already added (name + description string)
seen_combinations = set()

for key, df in all_dataframes.items():
    if isinstance(df, pd.DataFrame) and 'column_meta' in df.attrs:
        meta = df.attrs['column_meta']
        for col in df.columns:
            desc, unit = meta.get(col, ("No description found", "N/A"))

            # Create a unique fingerprint for this specific name/desc pair
            combination = f"{col}||{desc}"

            if combination not in seen_combinations:
                unique_pairs.append({
                    'Property': col,
                    'Description': desc,
                    'Unit': unit
                })
                seen_combinations.add(combination)

# Convert to DataFrame
df_unique_pairs = pd.DataFrame(unique_pairs).sort_values(
    by=['Property', 'Description'])

print(f"📊 Unique Engineering Pairs Found: {len(df_unique_pairs)}")
print("-" * 80)
print(df_unique_pairs.to_string(index=False))

📊 Unique Engineering Pairs Found: 67
--------------------------------------------------------------------------------
  Property                                              Description   Unit
         A                                          Area of section    mm2
        AL                         Painting surface per unit length m2.m-1
        Ct                               Torsional modulus constant    mm3
         D                                Outer diameter of section     mm
         G                                     Mass per unit length kg.m-1
        It                                         Torsion constant    mm4
        Iu                 Second moment of area about the u-u axis    mm4
        Iv                 Second moment of area about the v-v axis    mm4
        Iw                                         Warping constant    mm6
        Iy                 Second moment of area about the y-y axis    mm4
       Iyz                                       Centrifu

In [72]:
# 1. Identify descriptions that appear under multiple Property names
duplicate_descriptions = df_unique_pairs[df_unique_pairs.duplicated(
    'Description', keep=False)]

# 2. Group them to see which Properties are sharing the same definition
alias_groups = duplicate_descriptions.sort_values('Description')

print(f"🔍 Found {len(alias_groups)} properties with shared descriptions:")
print("-" * 80)
print(alias_groups.to_string(index=False))

🔍 Found 25 properties with shared descriptions:
--------------------------------------------------------------------------------
 Property                                              Description Unit
       u2               Distance of centre of gravity along y-axis   mm
       ys               Distance of centre of gravity along y-axis   mm
      y's               Distance of centre of gravity along y-axis   mm
       v3               Distance of centre of gravity along y-axis   mm
       v2               Distance of centre of gravity along y-axis   mm
       v1               Distance of centre of gravity along y-axis   mm
        v               Distance of centre of gravity along y-axis   mm
       u3               Distance of centre of gravity along y-axis   mm
       u1               Distance of centre of gravity along y-axis   mm
      z's               Distance of centre of gravity along z-axis   mm
       zs               Distance of centre of gravity along z-axis   mm
       

In [73]:
# import sqlite3
# import pandas as pd

# db_name = "steel_engineering_final.db"
# conn = sqlite3.connect(db_name)
# print(f"🚀 Final Migration: Case Sensitive + Updated Engineering Descriptions...")

# all_metadata_rows = []

# for key, df in all_dataframes.items():
#     if key.startswith('df_'):
#         table_name = key.replace('df_', 'sections_').lower()

#         # 1. Cast numeric data but preserve original casing (e.g., Iy, α)
#         df_sql = df.copy()
#         for col in df_sql.columns:
#             if col != 'Section_ID':
#                 df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')

#         # 2. Save to SQL
#         df_sql.to_sql(table_name, conn, if_exists='replace', index=False)

#         # 3. Save Context-Aware Metadata
#         if 'column_meta' in df.attrs:
#             for col_name, (desc, unit) in df.attrs['column_meta'].items():
#                 all_metadata_rows.append({
#                     'table_name': table_name,
#                     'column_name': col_name,
#                     'description': desc,
#                     'unit': unit
#                 })
#         print(f"  ✅ {table_name}: Migrated.")

# # Save the master dictionary
# df_dict = pd.DataFrame(all_metadata_rows)
# df_dict.to_sql('data_dictionary', conn, if_exists='replace', index=False)

# conn.close()
# print("\n🏆 DATABASE READY: All conflicts resolved, names preserved, and descriptions updated.")

In [74]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("steel_engineering_final.db")

# 1. Choose the table you want to inspect
target_table = 'sections_ipn'

# 2. Fetch the values
section_query = f"SELECT * FROM {target_table} ORDER BY RANDOM() LIMIT 1"
df_values = pd.read_sql(section_query, conn).T.reset_index()
df_values.columns = ['column_name', 'Value']

# 3. Fetch metadata dynamically for the SAME table
meta_query = f"SELECT column_name, description, unit FROM data_dictionary WHERE table_name = '{target_table}'"
df_meta = pd.read_sql(meta_query, conn)

# 4. Merge
report = pd.merge(df_values, df_meta, on='column_name', how='left')

# Clean up formatting
report['Result'] = report['Value'].astype(
    str) + " " + report['unit'].fillna("")

print(f"--- Inspection Report for: {target_table} ---")
print(report[['column_name', 'description', 'Result']].to_string(index=False))

conn.close()

--- Inspection Report for: sections_ipn ---
column_name                                              description           Result
 Section_ID                                Unique Section Identifier      IPN200 text
          h                                         Depth of section         200.0 mm
          b                                         Width of section          90.0 mm
         tf                                         Flange thickness          11.3 mm
         tw                                            Web thickness           7.5 mm
     r_root                                    Radius of root fillet           7.5 mm
      r_toe                              Radius of flange toe fillet           4.5 mm
         ys               Distance of centre of gravity along y-axis          45.0 mm
          d                         Depth of straight portion of web         159.1 mm
          G                                     Mass per unit length      26.2 kg.m-1
         A

### to deleto